In [4]:
!git clone https://github.com/Kaivalya-2005/mcq-solver.git
%cd mcq-solver

Cloning into 'mcq-solver'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 87 (delta 41), reused 73 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 332.84 KiB | 11.09 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/mcq-solver


In [5]:
!pip install uv
!uv sync

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 74.4 MB/s eta 0:00:00:00:0100:01
Using CPython 3.13.15
Creating virtual environment at: .venv
Resolved 119 packages in 1ms
Prepared 94 packages in 30.05s                                           
░░░░░░░░░░░░░░░░░░░░ [0/94] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 94 packages in 14.76s                             
 + accelerate==1.15.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + cuda-bindings==

In [6]:
import os
os.environ["MCQ_DATA_DIR"] = "/kaggle/input/competitions/smart-mcq-solver-challenge"  # check exact path in Data tab

In [7]:
!uv run python -c "import os, pandas as pd; p=os.environ['MCQ_DATA_DIR']; print(os.listdir(p)); print('Train:', pd.read_csv(os.path.join(p,'train.csv')).shape); print('Test:', pd.read_csv(os.path.join(p,'test.csv')).shape)"

['sample_submission.csv', 'train.csv', 'test.csv']
Train: (2000, 8)
Test: (500, 7)


In [13]:
%cd /kaggle/working/mcq-solver
!git pull origin main

/kaggle/working/mcq-solver
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 4), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 6.94 KiB | 2.31 MiB/s, done.
From https://github.com/Kaivalya-2005/mcq-solver
 * branch            main       -> FETCH_HEAD
   d203ccf..26e7a08  main       -> origin/main
Updating d203ccf..26e7a08
Fast-forward
 README.md                  |  15 ++-
 notebooks/mcq-solver.ipynb |   2 +-
 src/stage5_ensemble.py     | 294 +++++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 307 insertions(+), 4 deletions(-)
 create mode 100644 src/stage5_ensemble.py


In [13]:
!uv run python src/stage2_deberta_train.py

Training rows: 1608
Validation rows: 392
Test rows: 500
config.json: 100%|████████████████████████████| 579/579 [00:00<00:00, 2.78MB/s]
tokenizer_config.json: 100%|█████████████████| 52.0/52.0 [00:00<00:00, 273kB/s]

spm.model: downloading bytes:                              |  0.00B            
spm.model: downloading bytes: █████████████████████████████| 1.64MB,  162kB/s  
spm.model: reconstructing file: 100%|█████████████| 2.46MB / 2.46MB,  243kB/s  
/kaggle/working/mcq-solver/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

pytorch_model.bin: downloading bytes: ████████████████████▌|  364MB, 32.3MB/s  
pytorch_model.bin: reconstructing file:  72%|███▌ |  268MB /  371MB            
pytorch_model.bin: downloading bytes: █████████████████████|  371MB, 33.8MB/s  
pytorch_model.bin: reconstructing file: 100%|█████|  371MB /  371MB, 35.3MB/s  
Loading weights

In [8]:
%cd /kaggle/working/mcq-solver
%mkdir -p data_enriched

/kaggle/working/mcq-solver


In [11]:
%cp /kaggle/input/datasets/kaivalya24f1000791/data-enriched/train_with_context.csv /kaggle/working/mcq-solver/data_enriched/train_with_context.csv
%cp /kaggle/input/datasets/kaivalya24f1000791/data-enriched/test_with_context.csv /kaggle/working/mcq-solver/data_enriched/test_with_context.csv
%cp /kaggle/input/datasets/kaivalya24f1000791/data-enriched/llm_cache.json /kaggle/working/mcq-solver/data_enriched/llm_cache.json

In [14]:
!uv run python src/stage5_ensemble.py --allow-ambiguous-val-cache

Loading context-enriched datasets...
Applying leakage-free group-aware split (same as Stage 2/4)...
Train rows: 1608 | Val rows: 392 | Test rows: 500
Prompt-string overlap train/val: 0 (grouping is enforced by src.splits.clean_query)
Locating Stage 2 best checkpoint/model...
Using Stage 2 model: stage2_out/final
Running DeBERTa inference to get 5-way probabilities...
/kaggle/working/mcq-solver/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 2756.72it/s]
Loading Stage 4 cache and converting rankings to normalized score distributions...
Evaluating alpha sweep...

Alpha    MAP@3
0.0      0.6930
0.1      0.6943
0.2      0.6977
0.3      0.7011
0.4      0.7032
0.5      0.7041
0.6      0.7032
0.7      0.7032
0.8      0.7024
0.9      0.7007
1.0      0.4324

Summary
Best alpha: 0.5
Best ensemble MAP